# Evaluation set: Synthbuster + RAISE-1k (Section III-D)

Builds the primary evaluation set of the paper on a **CPU session (no GPU quota)**:

- **Synthetic:** Synthbuster [19] from Zenodo (record 10066460, CC BY-NC-SA 4.0): 1,000 images from each of 9 generators.
- **Authentic:** the 1,000 RAISE-1k photographs [20], downloaded as uncompressed TIFF from the RAISE site (non-commercial research use; cite the RAISE paper).
- Every image is **center-cropped to 336 × 336 without resizing**; images whose shorter side is below 336 px are **excluded and counted per generator**.
- For the recompression diagnostic, every crop is also saved **re-encoded once as JPEG at q ∈ {50, 60, 70, 80, 90, 95}** (Pillow/libjpeg, default 4:2:0 subsampling).

**Before running:**
1. Fill in the RAISE-1k form at http://loki.disi.unitn.it/RAISE/download.html (RAISE-1k → name, affiliation, e-mail, accept the terms) and download the **CSV** it gives you. It lists a `TIFF` URL for each photo.
2. Upload that CSV to Kaggle as a small private dataset and add it as **input** to this notebook.
3. Session: **Accelerator None**, **Internet On**. Then Save & Run All.

Outputs in `/kaggle/working/synthbuster_336/`: `orig/<generator>/*.png`, `jpeg_q<q>/<generator>/*.jpg`, `manifest.csv` (one row per source image, with exclusions), `exclusions.csv` (counts per generator), `build_info.json`. Make the output a dataset and attach it to the evaluation notebook.

## 1. Configuration

In [ ]:
import io, os, re, csv, json, time, zipfile, hashlib, subprocess, sys, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from PIL import Image

INPUT_ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working/synthbuster_336")
TMP = Path("/tmp/sb")                       # large scratch disk, not saved
ZENODO_URL = "https://zenodo.org/records/10066460/files/synthbuster.zip?download=1"
CROP = 336
QUALITIES = [50, 60, 70, 80, 90, 95]
N_THREADS = 8
LIMIT = None      # e.g. 20 for a quick test run (images per generator); None for the full build

Image.MAX_IMAGE_PIXELS = None               # RAISE photos are large
NTFY_TOPIC = ""   # live progress at https://ntfy.sh/<topic>; "" turns it off


def notify(msg):
    if NTFY_TOPIC:
        try:
            urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{NTFY_TOPIC}", data=str(msg)[:3500].encode(),
                                                          method="POST"), timeout=10).read()
        except Exception as e:
            print("notify failed:", type(e).__name__)


T0 = time.time()
DEADLINE = T0 + 10.5 * 3600   # stop starting RAISE downloads after this, so the session ends with its output saved
OUT.mkdir(parents=True, exist_ok=True); TMP.mkdir(parents=True, exist_ok=True)
notify("prep session started")

## 2. Helpers: center crop, save, recompress

In [ ]:
def to_rgb8(im):
    """PIL image -> 8-bit RGB array (handles 16-bit TIFFs)."""
    if im.mode in ("RGB", "L", "P", "RGBA", "CMYK"):
        return np.asarray(im.convert("RGB"))
    a = np.asarray(im)
    if a.dtype == np.uint16:
        a = (a >> 8).astype(np.uint8)
    if a.ndim == 2:
        a = np.stack([a] * 3, -1)
    return np.ascontiguousarray(a[..., :3])


def read_rgb8(data):
    try:
        with Image.open(io.BytesIO(data)) as im:
            return to_rgb8(im)
    except Exception:
        import tifffile  # fallback for TIFF layouts Pillow cannot read (e.g. 48-bit RGB)
        a = tifffile.imread(io.BytesIO(data))
        if a.dtype == np.uint16:
            a = (a >> 8).astype(np.uint8)
        return np.ascontiguousarray(a[..., :3])


def center_crop(a):
    h, w = a.shape[:2]
    y, x = (h - CROP) // 2, (w - CROP) // 2
    return a[y:y + CROP, x:x + CROP]


def process(data, generator, name, label, source):
    """Crop one image and write the PNG and its recompressed JPEG copies. Returns a manifest row."""
    row = {"generator": generator, "name": name, "label": label, "source": source}
    try:
        a = read_rgb8(data)
    except Exception as e:
        return {**row, "excluded": True, "reason": f"unreadable: {type(e).__name__}"}
    h, w = a.shape[:2]
    row.update(src_w=w, src_h=h)
    if min(h, w) < CROP:
        return {**row, "excluded": True, "reason": "short_side_lt_336"}
    img = Image.fromarray(center_crop(a))
    rel = f"{generator}/{Path(name).stem}"
    p = OUT / "orig" / f"{rel}.png"
    p.parent.mkdir(parents=True, exist_ok=True)
    img.save(p, format="PNG")
    for q in QUALITIES:
        j = OUT / f"jpeg_q{q}" / f"{rel}.jpg"
        j.parent.mkdir(parents=True, exist_ok=True)
        img.save(j, format="JPEG", quality=q)
    return {**row, "excluded": False, "reason": "", "path": f"orig/{rel}.png", "id": f"{generator}/{Path(name).stem}"}


def download(url, dst, retries=5):
    """Streaming download with resume (HTTP Range) and retries."""
    dst = Path(dst)
    for attempt in range(retries):
        try:
            have = dst.stat().st_size if dst.exists() else 0
            req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
            with urllib.request.urlopen(req, timeout=120) as r, open(dst, "ab" if have else "wb") as f:
                if have and getattr(r, "status", 206) == 200:   # server ignored Range: start over
                    f.truncate(0)
                done, t_last = have, time.time()
                while chunk := r.read(1 << 22):
                    f.write(chunk)
                    done += len(chunk)
                    if done > 1e9 and time.time() - t_last > 600:   # big files only, every 10 min
                        t_last = time.time()
                        print(f"  {dst.name}: {done / 1e9:.1f} GB", flush=True)
                        notify(f"prep: {dst.name} {done / 1e9:.1f} GB downloaded")
            return dst
        except Exception as e:
            print(f"  retry {attempt + 1}/{retries} {url[:80]}: {type(e).__name__} {e}")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"download failed: {url}")

## 3. Synthbuster (synthetic): download from Zenodo, crop every image

In [ ]:
zpath = TMP / "synthbuster.zip"
if not zpath.exists() or not zipfile.is_zipfile(zpath):
    print("downloading Synthbuster (12.4 GB) ..."); t = time.time()
    download(ZENODO_URL, zpath)
    print(f"  done in {(time.time() - t) / 60:.0f} min, {zpath.stat().st_size / 1e9:.1f} GB")
    notify(f"prep: Synthbuster zip downloaded ({zpath.stat().st_size / 1e9:.1f} GB in {(time.time() - t) / 60:.0f} min)")

IMG = re.compile(r"\.(png|jpe?g|webp|tiff?)$", re.I)
with zipfile.ZipFile(zpath) as z:
    members = [m for m in z.namelist() if IMG.search(m) and "__MACOSX" not in m]
# generator = the folder directly containing the image
gens = sorted({Path(m).parent.name for m in members})
print(f"{len(members)} images in {len(gens)} folders: {gens}")
if LIMIT:
    members = [m for g in gens for m in [x for x in members if Path(x).parent.name == g][:LIMIT]]


def do_member(m):
    with zipfile.ZipFile(zpath) as z:     # one handle per call: ZipFile is not thread-safe
        data = z.read(m)
    return process(data, Path(m).parent.name, Path(m).name, 1, "synthbuster")


t = time.time()
with ThreadPoolExecutor(N_THREADS) as ex:
    rows_fake = list(ex.map(do_member, members))
print(f"synthetic: {len(rows_fake)} processed in {(time.time() - t) / 60:.1f} min")
notify(f"prep: synthetic done, {sum(not r['excluded'] for r in rows_fake)} of {len(rows_fake)} kept; starting RAISE")

## 4. RAISE-1k (authentic): download each TIFF from the URL list, crop, delete

In [ ]:
def find_raise_csv():
    for dp, _, fs in os.walk(INPUT_ROOT, followlinks=True):
        for f in fs:
            if f.lower().endswith(".csv"):
                df = pd.read_csv(Path(dp) / f)
                if any(c.strip().upper() == "TIFF" for c in df.columns):
                    return Path(dp) / f, df
    raise FileNotFoundError("RAISE CSV with a 'TIFF' column not found under /kaggle/input: add it as input")


raise_csv, rdf = find_raise_csv()
tiff_col = next(c for c in rdf.columns if c.strip().upper() == "TIFF")
urls = [u.strip() for u in rdf[tiff_col].dropna().astype(str) if u.strip().startswith(("http", "file:"))]
print(f"{raise_csv.name}: {len(urls)} TIFF URLs")
if LIMIT:
    urls = urls[:LIMIT]


def do_raise(url):
    name = Path(urllib.request.urlparse(url).path).name or hashlib.md5(url.encode()).hexdigest()
    if time.time() > DEADLINE:
        return {"generator": "raise", "name": name, "label": 0, "source": "raise-1k",
                "excluded": True, "reason": "download_failed: session time limit"}
    tmp = TMP / "raise" / name
    tmp.parent.mkdir(parents=True, exist_ok=True)
    try:
        download(url, tmp)
        return process(tmp.read_bytes(), "raise", name, 0, "raise-1k")
    except Exception as e:
        return {"generator": "raise", "name": name, "label": 0, "source": "raise-1k",
                "excluded": True, "reason": f"download_failed: {type(e).__name__}"}
    finally:
        tmp.unlink(missing_ok=True)   # keep the disk free: TIFFs are ~30-50 MB each


t = time.time()
with ThreadPoolExecutor(min(N_THREADS, 6)) as ex:   # be gentle with the RAISE server
    rows_real = []
    for i, r in enumerate(ex.map(do_raise, urls)):
        rows_real.append(r)
        if i % 50 == 0:
            el = (time.time() - t) / 60
            msg = f"RAISE {i + 1}/{len(urls)} ({el:.0f} min, ~{el / (i + 1) * (len(urls) - i - 1):.0f} min left)"
            print("  " + msg, flush=True)
            notify("prep: " + msg)
print(f"authentic: {len(rows_real)} processed in {(time.time() - t) / 60:.1f} min")

## 5. Manifest, exclusion counts per generator, checks

In [ ]:
man = pd.DataFrame(rows_fake + rows_real)
man.to_csv(OUT / "manifest.csv", index=False)
ex = (man.groupby(["generator", "label"])
         .agg(total=("name", "size"), kept=("excluded", lambda s: int((~s.astype(bool)).sum())),
              excluded=("excluded", lambda s: int(s.astype(bool).sum())))
         .reset_index())
reasons = man[man.excluded.astype(bool)].groupby(["generator", "reason"]).size().rename("n").reset_index()
ex.to_csv(OUT / "exclusions.csv", index=False)
print(ex.to_string(index=False))
if len(reasons):
    print("\nexclusion reasons:\n" + reasons.to_string(index=False))

kept = man[~man.excluded.astype(bool)]
assert kept["id"].is_unique, "duplicate ids"
for _, r in kept.sample(min(20, len(kept)), random_state=0).iterrows():
    with Image.open(OUT / r["path"]) as im:
        assert im.size == (CROP, CROP), (r["path"], im.size)
    for q in QUALITIES:
        assert (OUT / f"jpeg_q{q}" / r["path"].replace("orig/", "", 1).replace(".png", ".jpg")).is_file()
print(f"\nkept {len(kept)} images: {int((kept.label == 1).sum())} synthetic, {int((kept.label == 0).sum())} authentic")
if (kept.label == 0).sum() < 0.9 * len(urls):
    print("WARNING: many RAISE photos missing; check the download_failed rows and rerun (finished files are kept)")

info = {"zenodo": ZENODO_URL, "raise_csv": raise_csv.name, "crop": CROP, "qualities": QUALITIES,
        "limit": LIMIT, "kept": int(len(kept)), "minutes": round((time.time() - T0) / 60, 1),
        "pillow": Image.__version__ if hasattr(Image, "__version__") else __import__("PIL").__version__}
(OUT / "build_info.json").write_text(json.dumps(info, indent=1))
print(json.dumps(info, indent=1))
notify("PREP DONE\n" + ex.to_string(index=False))